# Create an annotated BAL slide TIFF

**Select a slide → run the analysis → open the finished annotated TIFF**

The main output is a **full-resolution annotated TIFF** of the complete input slide. It contains segmented cell contours and matching coloured class labels, without confidence percentages. Macrophage particle estimates and neutrophil expert category predictions are included in the labels when applicable. Model probabilities remain in the CSV tables. The original scan remains unchanged; contours and text are rendered into a separate RGB TIFF, rather than removable annotation layers.

Cellpose-SAM → trained eight-class DinoBloom-Base classifier → particle rule / neutrophil expert → annotated BigTIFF.

The eight classes include **erythrocytes**. They remain in the cell table, total counts and TIFF overlays (`Ery`); only WBC-specific percentages exclude them.

Run the numbered cells with **Shift+Enter**, or use **Run All** after setting the slide path. Full-slide processing is the default. An optional short preview mode is available for the first trial. CSV files and biological summaries are additional outputs.

**First use:** follow `INSTALL.md`, copy the project's two trained classifier files into `models/`, and select the **BAL analysis (Python 3.11)** kernel. Cellpose downloads its built-in `cpsam` weights once if they are not already cached. Keep `bal_pipeline.py` beside this notebook. No new model training is performed.


## 1. Open the package folder

Start Jupyter in the extracted package folder. If needed, change `PROJECT_DIR` below. Windows paths can use forward slashes, for example `Path("D:/BAL_Lab")`.


In [ ]:
from pathlib import Path
import sys
import json

PROJECT_DIR = Path.cwd().resolve()
if not (PROJECT_DIR / "bal_pipeline.py").is_file():
    raise FileNotFoundError("Open Jupyter in the extracted package folder, or edit PROJECT_DIR above.")
sys.path.insert(0, str(PROJECT_DIR))

import pandas as pd
from IPython.display import display, Markdown, FileLink
from bal_pipeline import Config, Pipeline, CLASSES, EXPERT_CLASSES, discover_slides, load_samples, preflight

print("Project:", PROJECT_DIR)
print("Python kernel:", sys.executable)


## 2. Select the slide

Set `SLIDE_FILE` to the scan path. Alternatively, leave it empty and put **one** CZI/RGB TIFF in `input_slides/`. Set the results folder if it should be on another drive. Keep the two trained classifier files in `models/`, or change `MODEL_FOLDER`. Leave `CELLPOSE_MODEL="cpsam"` for automatic library caching; an explicit local file path is also supported for offline use.

`RUN_MODE = "full"` creates the complete annotated TIFF. For a small trial, use `"preview"` and select visited tile indices. The first tile can be an empty corner. A preview TIFF must not be used as a complete-slide result.


In [ ]:
SLIDE_FILE = ""       # Example: "D:/Slides/my_slide.czi"
OUTPUT_FOLDER = PROJECT_DIR / "analysis_results"
MODEL_FOLDER = PROJECT_DIR / "models"
CELLPOSE_MODEL = "cpsam"   # Cellpose downloads once and reuses its cache; or supply a local file path

RUN_MODE = "full"     # "full" = finished whole-slide TIFF; "preview" = partial trial
PREVIEW_TILES = 1
PREVIEW_START_TILE = 0

DEVICE = "auto"      # Cellpose: NVIDIA CUDA, then Apple MPS, otherwise CPU
CLASSIFIER_DEVICE = "same"  # use "cpu" to keep Cellpose on MPS but run the two classifiers on CPU
CLASSIFICATION_BATCH_SIZE = 8
CELLPOSE_BATCH_SIZE = 2
ENABLE_NEUTROPHIL_EXPERT = True

# Optional; not needed to create the annotated TIFF.
METADATA_CSV = None

config = Config(
    classifier_checkpoint=str(MODEL_FOLDER / "dinobloom_bal_production.pth"),
    expert_checkpoint=str(MODEL_FOLDER / "neutrophil_expert_finetuned.pth"),
    cellpose_checkpoint=str(CELLPOSE_MODEL),
    output_dir=str(OUTPUT_FOLDER),
    enable_expert=ENABLE_NEUTROPHIL_EXPERT,
    device=DEVICE,
    classifier_device=CLASSIFIER_DEVICE,
    batch_size=CLASSIFICATION_BATCH_SIZE,
    cellpose_batch_size=CELLPOSE_BATCH_SIZE,
    save_annotated_tiff=True,   # the required primary output
)


## 3. Check input, models and disk space

The check reads image headers and verifies the paths; it does not yet run the neural models. It prints separate devices for Cellpose and the classifiers. On a compatible Mac, `auto` selects **MPS**; explicit `DEVICE="mps"` is also supported. MPS uses float32 Cellpose inference. Some Cellpose mask-construction work remains on CPU. This version accepts a **single 8-bit RGB plane**, not fluorescence stacks or uint16 images. Review the displayed dimensions and CZI pixel spacing when available. Comparable staining and acquisition scale are needed for the trained models.

The image is decoded to a temporary disk-backed array. This reduces RAM pressure but needs SSD space. The models and segmentation still require working memory. CPU processing is supported but can be slow for a complete slide.


In [ ]:
if SLIDE_FILE:
    selected_paths = [Path(SLIDE_FILE).expanduser().resolve()]
else:
    selected_paths = discover_slides(PROJECT_DIR / "input_slides")
    if len(selected_paths) != 1:
        raise ValueError(f"Found {len(selected_paths)} slides in input_slides. Put exactly one there, or set SLIDE_FILE explicitly.")

samples = load_samples(selected_paths, METADATA_CSV)
image_info = preflight(config, samples)
display(image_info[["filename", "width", "height", "axes", "dtype", "tiles", "decoded_GiB", "pixel_size_um"]])
print("Mode:", RUN_MODE.upper())
print("Eight cell categories:", ", ".join(CLASSES))
print("Neutrophil categories:", ", ".join(EXPERT_CLASSES))


## 4. Generate the annotated TIFF

This step obtains Cellpose's `cpsam` weights from its cache (downloading once if needed), loads the two local trained classifiers, segments and classifies the image, then writes the TIFF. The first Cellpose download needs internet access; later use reuses the cached file. Progress is printed below. File hashing/model loading may take time before the first tile message. Do not close the notebook while it is running.

Identical completed runs are reused after their output checksums pass. An interrupted slide restarts in a new attempt folder; unfinished outputs are not presented as completed TIFFs. Input scans and weight files are not edited.


In [ ]:
pipeline = Pipeline(config)
result = pipeline.run(samples, mode=RUN_MODE, preview_tiles=PREVIEW_TILES, start_tile=PREVIEW_START_TILE)


## 5. Finished TIFF

The path and link below identify the main deliverable. The TIFF is **full resolution, tiled BigTIFF**, with cell-contour/text overlays. It has **no multiresolution pyramid**. Use a compatible image viewer for the full scan rather than loading all pixels into the notebook. Files outside the Jupyter project folder can be opened from the printed path in the operating system's file browser.


In [ ]:
batch_folder = Path(result["batch_folder"])
for run in result["runs"]:
    run_folder = Path(run["folder"])
    report = json.loads((run_folder / "RUN_REPORT.json").read_text(encoding="utf-8"))
    tiff_path = run_folder / "annotated_contours.tif"
    if report["status"] != "completed" or not tiff_path.is_file():
        raise RuntimeError("No completed annotated TIFF is available. Review the processing error above.")
    heading = "ANNOTATED WHOLE-SLIDE TIFF READY" if report["scope"] == "full" else "PARTIAL PREVIEW TIFF ONLY"
    display(Markdown(f"### {heading}"))
    print("Slide:", report["sample"]["filename"])
    print("Image:", report["image"]["width"], "×", report["image"]["height"], "pixels")
    print("Retained cells:", f"{report['retained_cells']:,}")
    print("Erythrocytes:", f"{int(run['summary']['Count_Erythrocyte']):,}")
    print("TIFF:", tiff_path)
    try:
        relative = tiff_path.relative_to(PROJECT_DIR)
    except ValueError:
        print("Output is outside the Jupyter folder: open the path in Explorer/Finder.")
    else:
        display(FileLink(str(relative), result_html_prefix="Open/download annotated TIFF: "))
    print("Run record:", run_folder / "RUN_REPORT.json")
print("TIFF path index:", batch_folder / "TIFF_OUTPUTS.csv")


## 6. Inspect representative original and annotated regions

These smaller PNG pairs make it easier to check the overlay. The full TIFF remains the primary output.

Each cell has one label in the same colour as its contour, for example `Mac` or `Ery`, with no percentage. Every macrophage with particles includes its detected count, for example `Mac+p | P:3` or `Mac+p | P:0`. `P` is the crop dark-structure estimate; a star in `Mac+p* | P:3` marks particle-rule reassignment. Other eligible cells also show a count when positive.

When the neutrophil expert is enabled, the label directly shows `Immature Neu`, `Mature Neu F+` or `Mature Neu F-`, instead of a generic `Neu` label and a second line. These labels retain the neutrophil contour colour. Without an expert assignment the label remains `Neu`. Original classes, classifier scores and expert probabilities remain in `cells.csv` for review.

Expert Siglec-F category names are image-based assignments to reference categories, not direct marker-expression measurements. Dark structures are a proxy, not a direct count of individual nanotubes. Cells touching tile boundaries are flagged in the cell CSV because boundary fragments can remain.


In [ ]:
from PIL import Image

for run in result["runs"]:
    folder = Path(run["folder"])
    for original_path in sorted((folder / "previews").glob("*_original.png")):
        annotated_path = original_path.with_name(original_path.name.replace("_original", "_annotated"))
        display(Markdown("**Original region**"))
        with Image.open(original_path) as image:
            image.thumbnail((1100, 700))
            display(image.copy())
        display(Markdown("**Annotated region**"))
        with Image.open(annotated_path) as image:
            image.thumbnail((1100, 700))
            display(image.copy())
    if report["retained_cells"] == 0:
        print("No retained cells. Inspect the original image; a preview may have selected an empty corner.")


## Additional outputs · optional

Each result folder also contains `cells.csv`, `neutrophil_cells.csv`, `slide_summary.csv` and `external_contours.json`. The cell table links coordinates, original/final classes, probabilities and dark-structure counts. CSV files can be opened in Excel. JSON stores external mask polygons in global image-pixel coordinates, x then y; it does not retain holes as a full instance-label raster.

No metadata is required for the TIFF. If you also need animal/group summaries, provide a CSV with `filename,animal_id,treatment,timepoint` and set `METADATA_CSV` in Step 2. The filename must match exactly. Preparations a/b from the same mouse must share one animal ID. Counts are pooled within animals before group mean ± sample SD is calculated. See `metadata_example.csv` and README.md.


In [ ]:
display(result["slide_statistics"][["Slide", "Scope", "Total_Detected_Cells", "Total_WBC",
                                    "Total_Macrophages", "Count_Neutrophil", "Count_Erythrocyte"]])
print("Additional batch tables:", batch_folder)
for run in result["runs"]:
    folder = Path(run["folder"])
    cells_table = pd.read_csv(folder / "cells.csv", nrows=10)
    display(cells_table[["Cell_ID", "Class", "Model_Predicted_Class", "Model_Confidence",
                         "Dark_Structure_Count", "Expert_Class", "Expert_Confidence"]].head(10))


## Handover and reproducibility

Keep `RUN_REPORT.json` with the TIFF. It records input/model hashes, the resolved Cellpose cache path, software versions, parameters, scope and output checksums. Cellpose may download its built-in weights on first use; images are not uploaded. The downloaded model hash is recorded without assuming it is identical to a historical project cache.

This is a portable version of the production pipeline with corrected global-coordinate duplicate checks and bounded crop batches. Independent reflection-padded core boundaries are retained, so some boundary-crossing cells can still be split. It is not a reconstruction of historical thesis masks or a promise of identical old counts.

Synthetic software checks do not validate real neural predictions or the scanner-specific CZI decoder. The original trained weights and a representative CZI must be checked on the target lab computer before routine use. Follow the acceptance steps in INSTALL.md. Model training/self-training remains outside this inference notebook.
